In [ ]:
# ── FINAL SUMMARY ────────────────────────────────────────────────────────────

print("\n" + "=" * 100)
print("EVALUATION COMPLETE")
print("=" * 100)
print(f"\nDataset: {len(df_results)} question pairs evaluated")
print(f"Judge Model: {JUDGE_MODEL}")
print(f"Language: {LANGUAGE}")
print(f"Evaluation Timestamp: {datetime.now(UTC).isoformat()}")
print(f"\nOutput Files:")
print(f"  • {OUTPUT_CSV} (detailed results)")
print(f"  • llm_judge_comparison_summary.csv (statistical summary)")
print(f"  • llm_judge_comparison_plots.png (visualizations)")
print("\nKey Findings:")
for metric in metrics_list:
    stats_row = comparison_stats[metric]
    delta = stats_row['mean_a'] - stats_row['mean_b']
    sig = "***" if stats_row['t_pval'] < 0.001 else "**" if stats_row['t_pval'] < 0.01 else "*" if stats_row['t_pval'] < 0.05 else "ns"
    winner = "Model A" if delta > 0 else "Model B" if delta < 0 else "Tie"
    print(f"  • {metric.capitalize()}: {winner} ({delta:+.3f}) [{sig}]")

In [ ]:
# ── SAVE DETAILED RESULTS TO CSV ──────────────────────────────────────────────

# Prepare output dataframe
df_output = df_results.copy()
df_output = df_output[[
    "query",
    "answer_a",
    "answer_b",
    "golden_answer",
    "a_usefulness", "a_accuracy", "a_conciseness",
    "b_usefulness", "b_accuracy", "b_conciseness",
    "better_variant",
    "judge_reason",
    "created_at",
]]

df_output.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"\n✓ Detailed results saved to: {OUTPUT_CSV}")
print(f"  Rows: {len(df_output)}")
print(f"  Columns: {len(df_output.columns)}")

# Summary CSV
df_summary.to_csv("llm_judge_comparison_summary.csv", index=False)
print(f"✓ Summary table saved to: llm_judge_comparison_summary.csv")

In [ ]:
# ── CREATE SUMMARY TABLE ──────────────────────────────────────────────────────

summary_data = []
for metric in metrics_list:
    stats_row = comparison_stats[metric]
    summary_data.append({
        "Metric": metric.capitalize(),
        "Model A Mean": f"{stats_row['mean_a']:.3f}",
        "Model B Mean": f"{stats_row['mean_b']:.3f}",
        "Δ Mean (A-B)": f"{stats_row['mean_a'] - stats_row['mean_b']:+.3f}",
        "t-pval": f"{stats_row['t_pval']:.4f}",
        "Significance": "***" if stats_row['t_pval'] < 0.001 else "**" if stats_row['t_pval'] < 0.01 else "*" if stats_row['t_pval'] < 0.05 else "ns",
        "Cohen's d": f"{stats_row['cohens_d']:.3f}",
    })

df_summary = pd.DataFrame(summary_data)
print("\n" + "=" * 100)
print("SUMMARY TABLE")
print("=" * 100)
print(df_summary.to_string(index=False))
print("\nLegend: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── VISUALIZATIONS ────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(16, 10))
fig.suptitle(
    "LLM-as-Judge: Model A (Google Gemini 3.5 Flash) vs Model B (OpenAI GPT-oss 120B)\n"
    "Comparison on 1-5 scales (Usefulness, Accuracy, Conciseness)",
    fontsize=14, weight="bold"
)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.3)

metrics_list = ["usefulness", "accuracy", "conciseness"]

for idx, metric in enumerate(metrics_list):
    col_a = f"a_{metric}"
    col_b = f"b_{metric}"
    
    # Boxplot
    ax1 = fig.add_subplot(gs[idx, 0])
    bp = ax1.boxplot(
        [df_results[col_a], df_results[col_b]],
        labels=["Model A", "Model B"],
        patch_artist=True,
        widths=0.6
    )
    bp["boxes"][0].set_facecolor("#534AB7")
    bp["boxes"][1].set_facecolor("#D85A30")
    ax1.set_ylabel(f"{metric.capitalize()} Score", fontsize=10)
    ax1.set_ylim(0.5, 5.5)
    ax1.grid(axis="y", alpha=0.3)
    ax1.set_title(f"{metric.capitalize()}: Boxplot", fontsize=10, weight="bold")
    
    # Histogram (overlaid)
    ax2 = fig.add_subplot(gs[idx, 1])
    ax2.hist(df_results[col_a], bins=5, alpha=0.6, label="Model A", color="#534AB7", edgecolor="black")
    ax2.hist(df_results[col_b], bins=5, alpha=0.6, label="Model B", color="#D85A30", edgecolor="black")
    ax2.set_xlabel("Score", fontsize=10)
    ax2.set_ylabel("Frequency", fontsize=10)
    ax2.legend(fontsize=9)
    ax2.set_title(f"{metric.capitalize()}: Distribution", fontsize=10, weight="bold")
    ax2.grid(axis="y", alpha=0.3)
    
    # Violin plot
    ax3 = fig.add_subplot(gs[idx, 2])
    parts = ax3.violinplot(
        [df_results[col_a], df_results[col_b]],
        positions=[1, 2],
        showmeans=True,
        showmedians=True,
    )
    ax3.set_xticks([1, 2])
    ax3.set_xticklabels(["Model A", "Model B"])
    ax3.set_ylabel(f"{metric.capitalize()} Score", fontsize=10)
    ax3.set_ylim(0.5, 5.5)
    ax3.grid(axis="y", alpha=0.3)
    ax3.set_title(f"{metric.capitalize()}: Violin Plot", fontsize=10, weight="bold")

plt.savefig("llm_judge_comparison_plots.png", dpi=150, bbox_inches="tight")
print("✓ Saved visualization: llm_judge_comparison_plots.png")
plt.show()

## Section 7: Generate Comparison Report

In [ ]:
from scipy import stats

# ── AGGREGATE SCORES BY METRIC ────────────────────────────────────────────────

metrics = ["usefulness", "accuracy", "conciseness"]
comparison_stats = {}

print("=" * 80)
print("STATISTICAL COMPARISON: MODEL A vs MODEL B")
print("=" * 80)

for metric in metrics:
    col_a = f"a_{metric}"
    col_b = f"b_{metric}"
    
    scores_a = df_results[col_a].values
    scores_b = df_results[col_b].values
    
    # Basic statistics
    mean_a = scores_a.mean()
    mean_b = scores_b.mean()
    std_a = scores_a.std()
    std_b = scores_b.std()
    median_a = np.median(scores_a)
    median_b = np.median(scores_b)
    
    # Paired t-test
    t_stat, t_pval = stats.ttest_rel(scores_a, scores_b)
    
    # Mann-Whitney U test (non-parametric)
    u_stat, u_pval = stats.mannwhitneyu(scores_a, scores_b, alternative='two-sided')
    
    # Effect size (Cohen's d for paired samples)
    diff = scores_a - scores_b
    cohens_d = diff.mean() / diff.std() if diff.std() > 0 else 0
    
    comparison_stats[metric] = {
        "mean_a": mean_a,
        "mean_b": mean_b,
        "std_a": std_a,
        "std_b": std_b,
        "median_a": median_a,
        "median_b": median_b,
        "t_stat": t_stat,
        "t_pval": t_pval,
        "u_stat": u_stat,
        "u_pval": u_pval,
        "cohens_d": cohens_d,
    }
    
    print(f"\n📊 METRIC: {metric.upper()}")
    print(f"  Model A (Gemini):     μ={mean_a:.3f}, σ={std_a:.3f}, median={median_a:.1f}")
    print(f"  Model B (GPT-oss):    μ={mean_b:.3f}, σ={std_b:.3f}, median={median_b:.1f}")
    print(f"  Difference (A - B):   Δμ={mean_a - mean_b:+.3f}")
    print(f"  Paired t-test:        t={t_stat:.3f}, p={t_pval:.4f} {'***' if t_pval < 0.001 else '**' if t_pval < 0.01 else '*' if t_pval < 0.05 else 'ns'}")
    print(f"  Mann-Whitney U:       U={u_stat:.1f}, p={u_pval:.4f}")
    print(f"  Cohen's d (effect):   {cohens_d:.3f}")

# ── PAIRWISE WINNER DISTRIBUTION ──────────────────────────────────────────────

print("\n" + "=" * 80)
print("PAIRWISE COMPARISON RESULTS")
print("=" * 80)

winner_counts = df_results["better_variant"].value_counts()
print(f"\nBetter variant (by judge):")
print(f"  Model A wins: {winner_counts.get('A', 0)} ({100*winner_counts.get('A', 0)/len(df_results):.1f}%)")
print(f"  Model B wins: {winner_counts.get('B', 0)} ({100*winner_counts.get('B', 0)/len(df_results):.1f}%)")

## Section 6: Compare Models with Statistical Analysis

In [ ]:
# ── RUN EVALUATION ────────────────────────────────────────────────────────────

print(f"Starting evaluation of {len(merged)} question pairs...")
print("This will call the LLM judge for each comparison. Progress shown below:\n")

results = []
failed_count = 0

for idx, row in merged.iterrows():
    if (idx + 1) % 10 == 0:
        print(f"Progress: {idx + 1}/{len(merged)} evaluated | Failed: {failed_count}")
    
    query = row["pytanie"]
    answer_a = row["answer_a"]
    answer_b = row["answer_b"]
    
    judge_scores = evaluate_pair(query, answer_a, answer_b)
    
    if judge_scores:
        result_row = {
            "query": query,
            "answer_a": answer_a,
            "answer_b": answer_b,
            "golden_answer": row.get("golden_answer", ""),
            "a_usefulness": judge_scores["a_usefulness"],
            "a_accuracy": judge_scores["a_accuracy"],
            "a_conciseness": judge_scores["a_conciseness"],
            "b_usefulness": judge_scores["b_usefulness"],
            "b_accuracy": judge_scores["b_accuracy"],
            "b_conciseness": judge_scores["b_conciseness"],
            "better_variant": judge_scores["better_variant"],
            "judge_reason": judge_scores["reason"],
            "created_at": datetime.now(UTC).isoformat(),
        }
        results.append(result_row)
    else:
        failed_count += 1
        print(f"  ✗ Failed for question {idx + 1}")
    
    # Small delay to avoid rate limits
    time.sleep(0.5)

print(f"\n✓ Evaluation complete!")
print(f"  Successful: {len(results)}")
print(f"  Failed: {failed_count}")

df_results = pd.DataFrame(results)
print(f"\nResults shape: {df_results.shape}")

## Section 4 & 5: Evaluate Model A and B Responses

In [ ]:
# ── LLM JUDGE CONFIGURATION ──────────────────────────────────────────────────

def evaluate_pair(query: str, answer_a: str, answer_b: str, max_retries: int = 2) -> dict | None:
    """
    Evaluate two answers using LLM judge.
    
    Args:
        query: Original question
        answer_a: Model A response
        answer_b: Model B response
        max_retries: Number of retry attempts
        
    Returns:
        Dict with scores or None if evaluation fails
    """
    try:
        result = judge_pair(
            query,
            answer_a,
            answer_b,
            judge_model=JUDGE_MODEL,
            language=LANGUAGE,
            temperature=0.0,
            max_tokens=400,
            max_retries=max_retries,
        )
        return {
            "a_usefulness": result.variant_a.usefulness,
            "a_accuracy": result.variant_a.accuracy,
            "a_conciseness": result.variant_a.conciseness,
            "b_usefulness": result.variant_b.usefulness,
            "b_accuracy": result.variant_b.accuracy,
            "b_conciseness": result.variant_b.conciseness,
            "better_variant": result.better_variant,
            "reason": result.reason,
        }
    except Exception as e:
        print(f"Error evaluating question: {str(e)[:100]}")
        return None

print("LLM Judge configuration:")
print(f"  Judge Model: {JUDGE_MODEL}")
print(f"  Language: {LANGUAGE}")
print(f"  Metrics: Usefulness, Accuracy, Conciseness (1-5 scale)")
print(f"  Temperature: 0.0 (deterministic)")
print(f"  Max tokens: 400")

## Section 3: Define LLM-as-Judge Evaluation Metrics

In [ ]:
# Remove rows with missing answers
merged = merged.dropna(subset=["answer_a", "answer_b"])
print(f"After removing missing values: {len(merged)} questions")

In [ ]:
# ── MERGE DATASETS BY QUESTION ────────────────────────────────────────────────

# Normalize column names
df_a.columns = df_a.columns.str.strip()
df_b.columns = df_b.columns.str.strip()
df_golden.columns = df_golden.columns.str.strip()

# Create merged dataset: question -> answer_a, answer_b, golden_answer
merged = df_a[["pytanie", "odpowiedz_wygenerowana"]].copy()
merged.columns = ["pytanie", "answer_a"]

# Merge with Model B
merged = merged.merge(
    df_b[["pytanie", "odpowiedz_wygenerowana"]].rename(columns={"odpowiedz_wygenerowana": "answer_b"}),
    on="pytanie",
    how="inner",
    indicator=False
)

# Merge with golden answers (optional)
merged = merged.merge(
    df_golden[["pytanie", "odpowiedz"]].rename(columns={"odpowiedz": "golden_answer"}),
    on="pytanie",
    how="left",
    indicator=False
)

print(f"Merged dataset: {len(merged)} common questions")
print(f"Questions with golden answer: {merged['golden_answer'].notna().sum()}")
print("\nSample merged row:")
print(merged.iloc[0])

## Section 2: Prepare Evaluation Data

In [ ]:
# Check columns
print(f"Model A columns: {df_a.columns.tolist()}")
print(f"Model B columns: {df_b.columns.tolist()}")
print(f"Golden columns: {df_golden.columns.tolist()}")

In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────

CSV_MODEL_A = "answers_google_gemini-3.5-flash_t0.2_p2_200.csv"
CSV_MODEL_B = "answers_openai_gpt-oss-120b_free_t0.2_p2_200.csv"
CSV_GOLDEN = "final_notebooklm_QA_stat_test.csv"

JUDGE_MODEL = "anthropic/claude-opus-4.7"  # LLM for judging (via OpenRouter)
OUTPUT_CSV = "llm_judge_results_a_vs_b.csv"

# Set language for judge prompts
LANGUAGE = "pl"

print("Loading CSV files...")
df_a = pd.read_csv(CSV_MODEL_A)
df_b = pd.read_csv(CSV_MODEL_B)
df_golden = pd.read_csv(CSV_GOLDEN)

print(f"✓ Model A (Gemini): {len(df_a)} questions")
print(f"✓ Model B (GPT-oss): {len(df_b)} questions")
print(f"✓ Golden set: {len(df_golden)} questions")

# Show sample
print("\n--- Model A Sample ---")
print(df_a.head(1).to_string())
print("\n--- Model B Sample ---")
print(df_b.head(1).to_string())
print("\n--- Golden Set Sample ---")
print(df_golden.head(1).to_string())

## Section 1: Load and Explore Data

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path
from datetime import datetime, UTC
import sys
import time

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent.parent.parent.parent))

from src.evaluation.llm_judge.judge import judge_pair, JudgeResult

# LLM-as-Judge Model Comparison: A vs B
## Pre-computed Responses Evaluation without API Overhead

This notebook evaluates two LLM models (Model A: Gemini 3.5 Flash, Model B: GPT-oss 120B) using already-generated answers stored in CSV files. We use LLM-as-Judge (1-5 scale) to score responses and compare them statistically.